In [ ]:
%pip install folium
%pip install pandas
%pip isntall geopandas
%pip install folium

In [ ]:
import geopandas as gpd
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from folium.features import GeoJsonTooltip

In [ ]:
# Load Electric Power Transmission Lines shapefile
all_sites = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\Electric_Substations\Electric_Substations.shp"
)

all_sites.head(5)

In [ ]:
all_sites.info()

In [ ]:
all_sites.describe()

In [ ]:
# remove -999999 values from the data
all_sites = all_sites[all_sites['MIN_VOLT'] >= 1]

In [ ]:
substations = all_sites[all_sites["TYPE"] == "SUBSTATION"]

In [ ]:
substations = substations[substations["MAX_VOLT"] != -999999]

In [ ]:
substations = substations[substations["STATUS"] == "IN SERVICE"]

In [ ]:
substations = substations.to_crs(epsg=4326)

In [ ]:
# create a basemap
m = folium.Map(location=[39.8283, -98.5795], zoom_start=5)

# add substations with MarkerCluster
marker_cluster = MarkerCluster(name="Substations").add_to(m)

# tooltip formatting for the individual substations
for _, row in substations.iterrows():
    tooltip_text = f"""
    <b>Name:</b> {row['NAME']}<br>
    <b>City:</b> {row['CITY']}<br>
    <b>State:</b> {row['STATE']}<br>
    <b>Status:</b> {row['STATUS']}<br>
    <b>Max Volt:</b> {row['MAX_VOLT']}
    """
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=tooltip_text,
        icon=folium.Icon(color="green", icon="bolt", prefix="fa")
    ).add_to(marker_cluster)

# load solar installations shapefile
solar_sites = gpd.read_file(
    r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\HPspSHP\uspvdb_v2_0_20240801.shp"
)

# reproject to WGS84 for folium
solar_sites = solar_sites.to_crs(epsg=4326)

# style function for solar polygons 
solar_style = lambda feature: {
    'fillColor': 'orange',
    'color': 'red',
    'weight': 1.5,
    'fillOpacity': 0.5
}

# define optional tooltip fields if they exist
solar_tooltip_fields = []
solar_tooltip_aliases = []

if "STATE" in solar_sites.columns:
    solar_tooltip_fields.append("STATE")
    solar_tooltip_aliases.append("State:")

if "NAME" in solar_sites.columns:
    solar_tooltip_fields.append("NAME")
    solar_tooltip_aliases.append("Site:")

# add solar layer
folium.GeoJson(
    solar_sites,
    name="High-Powered Solar Installations",
    style_function=solar_style,
    tooltip=GeoJsonTooltip(
        fields=solar_tooltip_fields,
        aliases=solar_tooltip_aliases,
        sticky=True
    ) if solar_tooltip_fields else None
).add_to(m)

# add title
title_html = '<h3 style="position:absolute;z-index:100000;left:10vw">High Voltage Substations and Solar Sites</h3>'
m.get_root().html.add_child(folium.Element(title_html))

# add layer control
folium.LayerControl().add_to(m)

# display map
m


save image

In [ ]:
m.save(r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\images\highvoltage_subtations&solarsites.html")